# 03. Forecast

ARIMA(statsmodels) 기반 시계열 예측. 각 상품에 대해 향후 N개월을 95% 신뢰구간과 함께 예측합니다.

**더 정교한 예측이 필요하면** `prophet`을 별도 설치(`pip install prophet`)하고 노트북 끝의 prophet 셀을 사용하세요. 기본은 ARIMA — 의존성 가벼움.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'sales_clean.csv', parse_dates=['date'])
FORECAST_MONTHS = 6

In [ ]:
def fit_and_forecast(series, periods=6, order=(1, 1, 1)):
    series = series.sort_index()
    model = ARIMA(series, order=order)
    fit = model.fit()
    fc = fit.get_forecast(steps=periods)
    mean = fc.predicted_mean
    ci = fc.conf_int(alpha=0.05)
    return mean, ci, fit

all_forecasts = []
fig, axes = plt.subplots(df['product_id'].nunique(), 1, figsize=(10, 4 * df['product_id'].nunique()), squeeze=False)
for ax_idx, (pid, g) in enumerate(df.groupby('product_id')):
    g = g.sort_values('date').set_index('date')
    series = g['units_sold'].asfreq('MS')
    mean, ci, fit = fit_and_forecast(series, periods=FORECAST_MONTHS)

    ax = axes[ax_idx][0]
    ax.plot(series.index, series.values, label='Actual', marker='o')
    ax.plot(mean.index, mean.values, label='Forecast', marker='s', linestyle='--', color='C1')
    ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.2, color='C1', label='95% CI')
    ax.set_title(f'{pid} — ARIMA{(1,1,1)} forecast (next {FORECAST_MONTHS} months)')
    ax.set_xlabel('Month')
    ax.set_ylabel('Units')
    ax.legend()
    ax.grid(True, alpha=0.3)

    out = pd.DataFrame({
        'date': mean.index,
        'product_id': pid,
        'forecast_units': mean.values.round(1),
        'ci_lower': ci.iloc[:, 0].values.round(1),
        'ci_upper': ci.iloc[:, 1].values.round(1),
    })
    all_forecasts.append(out)

fig.tight_layout()
fig.savefig(DATA_DIR / 'forecast.png', dpi=120)
plt.show()

In [ ]:
forecast_df = pd.concat(all_forecasts, ignore_index=True)
forecast_df.to_csv(DATA_DIR / 'forecast.csv', index=False)
print(f'Saved → {(DATA_DIR / "forecast.csv").resolve()}')
forecast_df

## 다음 단계

1. `forecast.png`와 `forecast.csv`를 Claude Desktop 채팅에 첨부.
2. `trend_narrative(si)` 프롬프트로 비즈니스 언어 narrative 생성.
3. `inventory.csv`(또는 `sample_inventory.csv`)와 합쳐 `action_recommendations(si)` 호출.

## (선택) Prophet으로 더 정교한 예측

`pip install prophet` 후 다음 셀을 사용하세요. 계절성·휴일 효과를 더 잘 모델링합니다.

In [ ]:
# from prophet import Prophet
# # 사용하려면 위 줄의 주석을 풀고, 아래 코드를 실행하세요.
# for pid, g in df.groupby('product_id'):
#     g = g.sort_values('date').rename(columns={'date': 'ds', 'units_sold': 'y'})[['ds', 'y']]
#     m = Prophet(yearly_seasonality=True)
#     m.fit(g)
#     future = m.make_future_dataframe(periods=FORECAST_MONTHS, freq='MS')
#     fcst = m.predict(future)
#     fig = m.plot(fcst); plt.title(pid); plt.show()